# Feature Extraction
Read each decomposed .npz file (u, v, omega), compute 60 features per file, save as features.csv.
All 7173 files processed: absent, present, unknown.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats
from pathlib import Path

PROJECT_ROOT   = Path(r"D:\sop")
DECOMPOSED_DIR = PROJECT_ROOT / "data" / "decomposed"
FEATURES_DIR   = PROJECT_ROOT / "data" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 1000  # signals were resampled to 1000 Hz in preprocessing

print("Ready.")
print(f"Decomposed dir : {DECOMPOSED_DIR}")
print(f"Features output: {FEATURES_DIR}")

Ready.
Decomposed dir : D:\sop\data\decomposed
Features output: D:\sop\data\features


In [2]:
# verify decomposed files are present across all 3 classes

for cls in ["absent", "present", "unknown"]:
    files = list((DECOMPOSED_DIR / cls).glob("*.npz"))
    print(f"{cls}: {len(files)} files")

total = sum(len(list((DECOMPOSED_DIR / cls).glob("*.npz"))) for cls in ["absent", "present", "unknown"])
print(f"\nTotal: {total}")

absent: 2391 files
present: 2391 files
unknown: 2391 files

Total: 7173


In [3]:
# feature extraction function
# inputs : u (K, T) oscillatory modes, v (T,) jump component, omega (K,) center frequencies
# output : dict of 60 features

def extract_features(u, v, omega, sr=SAMPLE_RATE):
    features = {}
    K = u.shape[0]
    N = u.shape[1]

    # mode energies computed first : reused in energy ratios and weighted frequency
    mode_energies = np.array([np.sum(u[k] ** 2) for k in range(K)])
    total_energy   = np.sum(mode_energies) + 1e-10

    for k in range(K):
        mode   = u[k]
        prefix = f"u{k+1}"

        # --- time domain ---

        # mean: average value of the mode
        # formula: sum(u_k) / T
        features[f"{prefix}_mean"] = float(np.mean(mode))

        # variance: spread of amplitude values
        # formula: mean((u_k - mean(u_k))^2)
        features[f"{prefix}_var"] = float(np.var(mode))

        # rms: effective amplitude / signal power
        # formula: sqrt(mean(u_k^2))
        features[f"{prefix}_rms"] = float(np.sqrt(np.mean(mode ** 2)))

        # zero crossing rate: how often the mode crosses zero per sample
        # formula: count(sign changes) / T
        signs = np.sign(mode)
        signs[signs == 0] = 1  # treat exact zero as positive to avoid spurious crossings
        features[f"{prefix}_zcr"] = float(np.sum(np.diff(signs) != 0) / N)

        # skewness: asymmetry of amplitude distribution
        # formula: mean((u_k - mean)^3) / std^3
        features[f"{prefix}_skew"] = float(scipy.stats.skew(mode))

        # kurtosis: peakedness of amplitude distribution (Fisher / excess, normal = 0)
        # formula: mean((u_k - mean)^4) / std^4 - 3
        features[f"{prefix}_kurt"] = float(scipy.stats.kurtosis(mode))

        # --- frequency domain ---
        # rfft gives positive frequencies only (0 to sr/2)
        fft_vals   = np.fft.rfft(mode)
        magnitudes = np.abs(fft_vals)
        freqs      = np.fft.rfftfreq(N, d=1.0 / sr)  # in Hz
        mag_sum    = np.sum(magnitudes) + 1e-10

        # dominant frequency: frequency bin with highest magnitude
        # formula: freqs[argmax(|FFT(u_k)|)]
        features[f"{prefix}_dom_freq"] = float(freqs[np.argmax(magnitudes)])

        # spectral centroid: centre of mass of the frequency spectrum
        # formula: sum(freqs * |FFT|) / sum(|FFT|)
        centroid = float(np.sum(freqs * magnitudes) / mag_sum)
        features[f"{prefix}_spectral_centroid"] = centroid

        # spectral bandwidth: spread around the centroid
        # formula: sqrt(sum((freqs - centroid)^2 * |FFT|) / sum(|FFT|))
        features[f"{prefix}_spectral_bandwidth"] = float(
            np.sqrt(np.sum((freqs - centroid) ** 2 * magnitudes) / mag_sum)
        )

        # spectral entropy: how spread energy is across frequencies
        # formula: -sum(p * log2(p)) where p = |FFT|^2 / sum(|FFT|^2)
        power      = magnitudes ** 2
        power_norm = power / (np.sum(power) + 1e-10)
        features[f"{prefix}_spectral_entropy"] = float(
            -np.sum(power_norm * np.log2(power_norm + 1e-10))
        )

        # --- energy ---

        # mode energy: total power in this oscillatory component
        # formula: sum(u_k^2)
        features[f"{prefix}_energy"] = float(mode_energies[k])

        # energy ratio: fraction of total oscillatory power held by this mode
        # formula: energy(u_k) / sum(energy(u_1..u_K))
        features[f"{prefix}_energy_ratio"] = float(mode_energies[k] / total_energy)

    # --- omega features ---

    # raw center frequencies for each mode (radians per sample)
    for k in range(K):
        features[f"omega{k+1}"] = float(omega[k])

    # energy-weighted mean frequency: dominant frequency weighted by mode power
    # formula: sum(omega_k * energy_k) / sum(energy_k)
    features["omega_weighted_mean"] = float(np.sum(omega * mode_energies) / total_energy)

    # frequency spread: range between lowest and highest mode frequency
    # formula: omega[K-1] - omega[0]
    features["omega_spread"] = float(omega[-1] - omega[0])

    # --- jump features (from v) ---

    diff_v   = np.diff(v)
    abs_diff = np.abs(diff_v)
    threshold = 3.0 * np.std(diff_v)  # jumps defined as derivatives exceeding 3 std
    jump_mask = abs_diff > threshold

    # number of jumps: count of abrupt discontinuities above threshold
    features["v_num_jumps"] = int(np.sum(jump_mask))

    # jump magnitude mean: average size of detected jumps
    features["v_jump_mean"] = float(np.mean(abs_diff[jump_mask])) if jump_mask.any() else 0.0

    # jump magnitude max: largest single abrupt change in the signal
    features["v_jump_max"] = float(np.max(abs_diff))

    # jump magnitude std: variability in jump sizes
    features["v_jump_std"] = float(np.std(abs_diff[jump_mask])) if jump_mask.any() else 0.0

    # total variation: total accumulated abrupt change across entire signal
    # formula: sum(|v[i+1] - v[i]|)
    features["v_total_variation"] = float(np.sum(abs_diff))

    # jump energy: total power in the jump component
    # formula: sum(v^2)
    features["v_energy"] = float(np.sum(v ** 2))

    return features


print("Feature extraction function defined.")
print("Features per file: 60")
print("  u time-domain      : 6 x 4 modes = 24")
print("  u frequency-domain : 4 x 4 modes = 16")
print("  u energy           : 2 x 4 modes =  8")
print("  omega              :              =  6")
print("  v jump             :              =  6")

Feature extraction function defined.
Features per file: 60
  u time-domain      : 6 x 4 modes = 24
  u frequency-domain : 4 x 4 modes = 16
  u energy           : 2 x 4 modes =  8
  omega              :              =  6
  v jump             :              =  6


In [4]:
# dry run on one file from each class to confirm features compute correctly

for cls in ["absent", "present", "unknown"]:
    sample_file = sorted((DECOMPOSED_DIR / cls).glob("*.npz"))[0]
    d = np.load(sample_file)
    feats = extract_features(d["u"], d["v"], d["omega"])
    print(f"{cls} : {sample_file.name}")
    print(f"  feature count : {len(feats)}")
    print(f"  any NaN       : {any(np.isnan(v) for v in feats.values())}")
    print(f"  any Inf       : {any(np.isinf(v) for v in feats.values() if isinstance(v, float))}")
    print()

absent : a100_AV_decomposed.npz
  feature count : 60
  any NaN       : False
  any Inf       : False

present : p100_AV_decomposed.npz
  feature count : 60
  any NaN       : False
  any Inf       : False

unknown : u10_AV_decomposed.npz
  feature count : 60
  any NaN       : False
  any Inf       : False



In [5]:
# run feature extraction on all 7173 files
# processes absent → present → unknown in order
# prints progress every 500 files

rows   = []
errors = []

for cls in ["absent", "present", "unknown"]:
    # label assignment: absent=0, present=1, unknown=2
    label_map = {"absent": 0, "present": 1, "unknown": 2}
    label = label_map[cls]

    npz_files = sorted((DECOMPOSED_DIR / cls).glob("*.npz"))
    print(f"Processing {cls}: {len(npz_files)} files")

    for i, fpath in enumerate(npz_files):
        try:
            d     = np.load(fpath)
            feats = extract_features(d["u"], d["v"], d["omega"])

            feats["file"]  = fpath.stem
            feats["class"] = cls
            feats["label"] = label
            rows.append(feats)

        except Exception as e:
            errors.append({"file": fpath.stem, "class": cls, "error": str(e)})

        if (i + 1) % 500 == 0:
            print(f"  {i + 1}/{len(npz_files)} done")

    print(f"  {len(npz_files)}/{len(npz_files)} done")

print(f"\nTotal extracted : {len(rows)}")
print(f"Errors          : {len(errors)}")
if errors:
    for e in errors:
        print(f"  {e['class']}/{e['file']}: {e['error']}")

Processing absent: 2391 files
  500/2391 done
  1000/2391 done
  1500/2391 done
  2000/2391 done
  2391/2391 done
Processing present: 2391 files
  500/2391 done
  1000/2391 done
  1500/2391 done
  2000/2391 done
  2391/2391 done
Processing unknown: 2391 files
  500/2391 done
  1000/2391 done
  1500/2391 done
  2000/2391 done
  2391/2391 done

Total extracted : 7173
Errors          : 0


In [6]:
# build dataframe and reorder columns: file, class, label first, then all 60 features

df = pd.DataFrame(rows)

meta_cols    = ["file", "class", "label"]
feature_cols = [c for c in df.columns if c not in meta_cols]
df           = df[meta_cols + feature_cols]

print(f"Shape          : {df.shape}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Class counts   :")
print(df["class"].value_counts().to_string())
print(f"\nSample (first 3 rows, first 10 feature columns):")
print(df[meta_cols + feature_cols[:10]].head(3).to_string(index=False))

Shape          : (7173, 63)
Feature columns: 60
Class counts   :
class
absent     2391
present    2391
unknown    2391

Sample (first 3 rows, first 10 feature columns):
              file  class  label      u1_mean   u1_var   u1_rms   u1_zcr   u1_skew  u1_kurt  u1_dom_freq  u1_spectral_centroid  u1_spectral_bandwidth  u1_spectral_entropy
a100_AV_decomposed absent      0 9.848693e-14 0.000164 0.012792 0.106855  0.033353 8.042188    51.031200             59.247283              38.967179             9.170230
a100_MV_decomposed absent      0 5.446652e-12 0.000387 0.019664 0.100199 -0.002383 5.491796    49.086464             61.100212              50.319038             8.940403
a101_AV_decomposed absent      0 1.551149e-13 0.000297 0.017246 0.067544  0.002557 5.066671    36.989136             43.440635              44.168741             8.286918


In [7]:
# check for NaN and Inf in feature columns

nan_count = df[feature_cols].isna().sum().sum()
inf_count = np.isinf(df[feature_cols].values).sum()

print(f"NaN values : {nan_count}")
print(f"Inf values : {inf_count}")

if nan_count > 0:
    print("\nColumns with NaN:")
    print(df[feature_cols].isna().sum()[df[feature_cols].isna().sum() > 0])

if inf_count > 0:
    print("\nColumns with Inf:")
    inf_cols = [c for c in feature_cols if np.isinf(df[c].values).any()]
    print(inf_cols)

NaN values : 0
Inf values : 0


In [8]:
# save full feature table to data/features/features.csv
# all 7173 rows, all classes including unknown

output_path = FEATURES_DIR / "features.csv"
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows  : {len(df)}")
print(f"Cols  : {len(df.columns)}  (3 meta + 60 features)")
print(f"Size  : {output_path.stat().st_size / 1024 / 1024:.1f} MB")

Saved: D:\sop\data\features\features.csv
Rows  : 7173
Cols  : 63  (3 meta + 60 features)
Size  : 8.2 MB


In [9]:
# reload from disk and verify

df_check = pd.read_csv(FEATURES_DIR / "features.csv")

print(f"Reloaded shape : {df_check.shape}")
print(f"Classes        : {df_check['class'].value_counts().to_dict()}")
print(f"Feature columns: {[c for c in df_check.columns if c not in ['file','class','label']]}")

Reloaded shape : (7173, 63)
Classes        : {'absent': 2391, 'present': 2391, 'unknown': 2391}
Feature columns: ['u1_mean', 'u1_var', 'u1_rms', 'u1_zcr', 'u1_skew', 'u1_kurt', 'u1_dom_freq', 'u1_spectral_centroid', 'u1_spectral_bandwidth', 'u1_spectral_entropy', 'u1_energy', 'u1_energy_ratio', 'u2_mean', 'u2_var', 'u2_rms', 'u2_zcr', 'u2_skew', 'u2_kurt', 'u2_dom_freq', 'u2_spectral_centroid', 'u2_spectral_bandwidth', 'u2_spectral_entropy', 'u2_energy', 'u2_energy_ratio', 'u3_mean', 'u3_var', 'u3_rms', 'u3_zcr', 'u3_skew', 'u3_kurt', 'u3_dom_freq', 'u3_spectral_centroid', 'u3_spectral_bandwidth', 'u3_spectral_entropy', 'u3_energy', 'u3_energy_ratio', 'u4_mean', 'u4_var', 'u4_rms', 'u4_zcr', 'u4_skew', 'u4_kurt', 'u4_dom_freq', 'u4_spectral_centroid', 'u4_spectral_bandwidth', 'u4_spectral_entropy', 'u4_energy', 'u4_energy_ratio', 'omega1', 'omega2', 'omega3', 'omega4', 'omega_weighted_mean', 'omega_spread', 'v_num_jumps', 'v_jump_mean', 'v_jump_max', 'v_jump_std', 'v_total_variation', 